In [30]:
import pandas as pd
import numpy as np

In [31]:
df = pd.read_csv('Titanic-Dataset.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [32]:
df.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [34]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [35]:
df1= df.Cabin.isna().sum()
df1

np.int64(687)

In [36]:
df2 = df.drop(["Cabin", "Name", "Ticket"], axis=1)
df2.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.0,1,0,7.2500,S
1,2,1,1,female,38.0,1,0,71.2833,C
2,3,1,3,female,26.0,0,0,7.9250,S
3,4,1,1,female,35.0,1,0,53.1000,S
4,5,0,3,male,35.0,0,0,8.0500,S


In [37]:
df2.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Sex              0
Age            177
SibSp            0
Parch            0
Fare             0
Embarked         2
dtype: int64

In [38]:
df2["Age"] = df2.Age.fillna(df2.Age.mean())

In [39]:
df2.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       2
dtype: int64

In [40]:
df2 = df2.dropna(subset=["Embarked"])
df2.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
dtype: int64

In [ ]:
from sklearn.preprocessing import LabelEncoder

# LabelEncode Sex (binary: female/male -> 0/1)
le = LabelEncoder()
df2["Sex"] = le.fit_transform(df2["Sex"])
print(dict(zip(le.classes_, le.transform(le.classes_))))
df2.head()

# --- What the line above does, piece by piece ---
print("classes_ :", le.classes_)            # unique labels found, sorted: [female male]
print("encoded  :", le.transform(le.classes_))  # their numbers: [0 1]
# zip() pairs them: [(female,0), (male,1)] -> dict() makes it readable
print("mapping  :", {label: int(code) for label, code in zip(le.classes_, le.transform(le.classes_))})
print("check    : male ->", le.transform(["male"]), "| female ->", le.transform(["female"]))


In [42]:
from sklearn.preprocessing import OneHotEncoder

# One-Hot Encode Embarked (C/Q/S -> separate 0/1 columns)
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
embarked_encoded = ohe.fit_transform(df2[["Embarked"]])
embarked_cols = ohe.get_feature_names_out(["Embarked"])
print("Categories:", list(embarked_cols))
df_emb = pd.DataFrame(embarked_encoded, columns=embarked_cols, index=df2.index)
df2 = pd.concat([df2.drop(columns=["Embarked"]), df_emb], axis=1)
df2.head()

Categories: ['Embarked_C', 'Embarked_Q', 'Embarked_S']


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,1,22.0,1,0,7.2500,0.0,0.0,1.0
1,2,1,1,0,38.0,1,0,71.2833,1.0,0.0,0.0
2,3,1,3,0,26.0,0,0,7.9250,0.0,0.0,1.0
3,4,1,1,0,35.0,1,0,53.1000,0.0,0.0,1.0
4,5,0,3,1,35.0,0,0,8.0500,0.0,0.0,1.0


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- What the data looks like BEFORE plotting ---
print("Age describe:")  # count/mean/std/min/max of Age
print(df2["Age"].describe().to_string())
print("Median Age:", df2["Age"].median())  # median vs mean shows skew
print("Bins = 30 -> each bar covers ~ (max-min)/30 years")

# Age distribution (Matplotlib + Seaborn)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1 row, 2 plots side-by-side
print("Figure size:", fig.get_size_inches(), "| Axes:", len(axes))

# 1. Matplotlib histogram
axes[0].hist(df2["Age"], bins=30, color="skyblue", edgecolor="black")  # 30 bars
axes[0].set_title("Age Distribution (Matplotlib)")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")
mean_age = df2["Age"].mean()  # red dashed center line
print("Mean Age:", round(float(mean_age), 2))
axes[0].axvline(mean_age, color="red", linestyle="--", label=f"Mean: {mean_age:.1f}")
axes[0].legend()

# 2. Seaborn histplot with KDE
sns.histplot(df2["Age"], bins=30, kde=True, color="teal", ax=axes[1])  # kde adds smooth curve
print("KDE = smooth density curve, peaks = common ages")
axes[1].set_title("Age Distribution with KDE (Seaborn)")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

# Optional: boxplot to spot outliers
print("Box = 25%-75 pct, line inside = median, whiskers/dots = outliers")
plt.figure(figsize=(8, 2))
sns.boxplot(x=df2["Age"], color="lightcoral")
plt.title("Age Boxplot")
plt.xlabel("Age")
plt.show()
print("Outlier check: min =", df2["Age"].min(), "| max =", df2["Age"].max())


In [ ]:
# Save cleaned dataset
print("Shape before save:", df2.shape)  # expect (889, 11)
print("Columns:", list(df2.columns))
print("Missing values:")  # all must be 0
print(df2.isna().sum().to_string())
df2.to_csv("cleaned_dada_titanic.csv", index=False)  # index=False avoids extra col
print("Saved: cleaned_dada_titanic.csv with shape", df2.shape)
print(df2.head().to_string())
# Verify: read it back to prove the file is correct
import pandas as pd
check = pd.read_csv("cleaned_dada_titanic.csv")
print("Reload check:", check.shape, "| cols match:", list(check.columns) == list(df2.columns))
